# working with databases (sqlite3)
* until now, if we wanted to keep data around, we used plain text files, json or csv (see lesson 05)
* a **relational database** organizes data into **tables** (think: a spreadsheet), each table has **rows** (one record) and **columns** (one field of that record, with a fixed datatype)
* every table usually has a **primary key** - a column (often just a number) that uniquely identifies each row, so we can find/update/delete exactly that one row again
* why use a real database instead of a csv/json file? multiple programs/users can read and write **concurrently** without corrupting the file, we can **query** for exactly the data we need (instead of loading and filtering everything in python ourselves), and we can express **relationships** between different kinds of data (e.g. "this book belongs to that author")
* `sqlite3` ships built into python (no pip install needed!) - its a lightweight, **file-based** (or, as we will mostly use it here, purely **in-memory**) database engine, perfect for learning SQL basics and for small real apps
* real, bigger production web apps usually use a proper database **server** instead, like PostgreSQL or MySQL, which run as their own separate process and can handle many concurrent connections over a network - but the SQL you write and the concepts you learn here transfer almost 1:1

## connecting to a database
* `sqlite3.connect(...)` opens (or creates, if it doesnt exist yet) a database and gives us a `Connection` object
* `connection.cursor()` gives us a `Cursor` - the object we actually use to run SQL statements and fetch results back
* important: sqlite3 does **not** auto-commit by default, changes only become permanent once we call `.commit()`
* using the connection as a **context manager** (`with connection:`) is the recommended way - it commits automatically for us when the block ends without error (and rolls back if an exception happened)

In [1]:
import sqlite3  # sqlite3 is part of the python standard library, no pip install needed

# we connect to an IN-MEMORY database for this whole lesson
# this means the database only lives in RAM and disappears the moment we close the connection
# perfect for learning/exercises: we never leave any .db file lying around on disk afterwards
connection = sqlite3.connect(":memory:")  # use a real filename like "mydata.db" instead if you want it to persist to disk

# the cursor is what we call .execute() on to actually run SQL, and later fetch results from
cursor = connection.cursor()

# lets create a tiny scratch table just to demonstrate committing, we drop it again right after
cursor.execute("CREATE TABLE scratch (id INTEGER)")  # a minimal one-column table, just for this demo
cursor.execute("INSERT INTO scratch (id) VALUES (1)")  # this change only lives in the current "transaction" so far
connection.commit()  # manual commit - we explicitly tell sqlite "yes, make this permanent now"
print("insert committed manually")

# the with-statement version does the exact same commit for us automatically
with connection:  # entering the with-block starts a transaction
    cursor.execute("INSERT INTO scratch (id) VALUES (2)")  # another change, not yet permanent
    # no connection.commit() needed here - leaving the "with" block without an error commits it for us
print("insert committed automatically by the with-block")

# note: "with connection:" only handles commit/rollback, it does NOT close the connection for us
# we still call connection.close() ourselves, once, at the very end of the notebook

cursor.execute("DROP TABLE scratch")  # cleaning up our little demo table again, we dont need it anymore
connection.commit()

insert committed manually
insert committed automatically by the with-block


## creating tables
* `CREATE TABLE IF NOT EXISTS ...` creates a new table, the "IF NOT EXISTS" part makes it safe to re-run without an error if the table is already there
* every column gets a name and a type (sqlite's main types: `INTEGER`, `TEXT`, `REAL` (floating point), `BLOB` (raw bytes))
* `PRIMARY KEY` marks the column that uniquely identifies each row - in sqlite, an `INTEGER PRIMARY KEY` column even auto-increments for us if we dont provide a value
* we set up two related tables here already: `authors` and `books`, where every book "belongs to" one author via a `author_id` column - more on this relationship later in the JOIN section

In [2]:
# the authors table: one row per author, id is the primary key
with connection:
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS authors (
            id INTEGER PRIMARY KEY,     -- unique id, auto-increments if we dont pass a value
            name TEXT NOT NULL          -- "NOT NULL" means this column must always have a value
        )
    ''')

# the books table: one row per book, and it references an author via author_id
with connection:
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS books (
            id INTEGER PRIMARY KEY,                    -- unique id for the book
            title TEXT NOT NULL,                        -- the books title, required
            year INTEGER,                                -- publishing year, optional (can be NULL)
            author_id INTEGER REFERENCES authors(id)     -- this column "points" to a row in the authors table
        )
    ''')

print("authors and books tables created")

authors and books tables created


## inserting rows
* `INSERT INTO table (...) VALUES (?, ?, ...)` adds a new row
* the `?` are **placeholders** - we pass the actual values separately as a tuple, sqlite3 inserts them safely for us
* **never** build a SQL string yourself with an f-string/`.format()` when any part of it comes from user input - this is called **SQL injection** and its one of the most common real-world security bugs, because a malicious user could type something like `Alex'); DROP TABLE authors; --` as their "name" and suddenly run their own SQL instead of just storing text
* parameterized queries (the `?` placeholders) are the safe way - sqlite3 escapes the values correctly for us, no matter what weird characters are inside

In [3]:
user_supplied_name = "Alex O'Brien"  # note the apostrophe - a classic case that breaks naive string building

# DANGEROUS - never do this with real user input! shown here only as a comment, NOT executed:
# cursor.execute(f"INSERT INTO authors (name) VALUES ('{user_supplied_name}')")
# the apostrophe in "O'Brien" would already break this SQL string, and with crafted input
# an attacker could inject arbitrary extra SQL commands - this is called SQL injection

# SAFE - the parameterized version, this is the one we actually run:
with connection:
    cursor.execute(
        "INSERT INTO authors (name) VALUES (?)",  # the "?" is a placeholder, not part of the string itself
        (user_supplied_name,),                      # the actual value, passed separately as a tuple
    )

print("inserted author safely, id was:", cursor.lastrowid)  # lastrowid gives us the auto-generated primary key of the last insert

inserted author safely, id was: 1


## inserting many rows at once
* `executemany()` runs the same SQL statement once per tuple in a list - much more convenient (and faster) than calling `.execute()` in a python for-loop

In [4]:
# a list of tuples, one tuple per author we still want to add
more_authors = [
    ("Ursula K. Le Guin",),   # note the trailing comma - this makes it a one-element tuple, not just a string in brackets
    ("Isaac Asimov",),
    ("Terry Pratchett",),
]

with connection:
    # executemany() loops through the list for us and runs the INSERT once per tuple
    cursor.executemany("INSERT INTO authors (name) VALUES (?)", more_authors)

print("authors table now has", cursor.execute("SELECT COUNT(*) FROM authors").fetchone()[0], "rows")

# now lets add some books the same way, each one referencing an author_id we look up first
le_guin_id = cursor.execute("SELECT id FROM authors WHERE name = ?", ("Ursula K. Le Guin",)).fetchone()[0]
asimov_id = cursor.execute("SELECT id FROM authors WHERE name = ?", ("Isaac Asimov",)).fetchone()[0]
pratchett_id = cursor.execute("SELECT id FROM authors WHERE name = ?", ("Terry Pratchett",)).fetchone()[0]

books_to_insert = [
    ("The Left Hand of Darkness", 1969, le_guin_id),  # (title, year, author_id) - matches the column order
    ("The Dispossessed", 1974, le_guin_id),
    ("Foundation", 1951, asimov_id),
    ("I, Robot", 1950, asimov_id),
    ("Guards! Guards!", 1989, pratchett_id),
    ("Mort", 1987, pratchett_id),
]

with connection:
    cursor.executemany("INSERT INTO books (title, year, author_id) VALUES (?, ?, ?)", books_to_insert)

print("books table now has", cursor.execute("SELECT COUNT(*) FROM books").fetchone()[0], "rows")

authors table now has 4 rows
books table now has 6 rows


## querying data
* `SELECT columns FROM table WHERE condition ORDER BY column` is the classic read query
* `WHERE` filters rows, `ORDER BY` sorts the result (add `DESC` for descending order)
* after `.execute()`, we can grab the results with `.fetchone()` (a single row, or `None` if there is none), `.fetchall()` (a list of all matching rows), or by directly iterating over the cursor itself (memory-friendly for big result sets, since it doesnt load everything at once)

In [5]:
# .fetchone() - grabs just the next single row (here: the first, since we only expect one match)
cursor.execute("SELECT title, year FROM books WHERE title = ?", ("Foundation",))
one_row = cursor.fetchone()
print("fetchone result:", one_row)  # a plain tuple by default: (title, year)

# .fetchall() - grabs every remaining matching row at once, as a list of tuples
cursor.execute("SELECT title, year FROM books WHERE year < ? ORDER BY year", (1970,))  # only older books, oldest first
old_books = cursor.fetchall()
print("fetchall result:", old_books)

# iterating the cursor directly - no explicit fetch call needed, python just loops row by row
cursor.execute("SELECT title, year FROM books ORDER BY year DESC")  # newest book first
print("iterating the cursor directly:")
for title, year in cursor:  # each row unpacks just like a normal tuple
    print(" -", title, f"({year})")

fetchone result: ('Foundation', 1951)
fetchall result: [('I, Robot', 1950), ('Foundation', 1951), ('The Left Hand of Darkness', 1969)]
iterating the cursor directly:
 - Guards! Guards! (1989)
 - Mort (1987)
 - The Dispossessed (1974)
 - The Left Hand of Darkness (1969)
 - Foundation (1951)
 - I, Robot (1950)


## updating and deleting rows
* `UPDATE table SET column = value WHERE condition` changes existing rows
* `DELETE FROM table WHERE condition` removes rows
* **always** double check your `WHERE` clause - leaving it out updates/deletes **every single row** in the table, which is a classic (and painful) real-world mistake

In [6]:
# UPDATE - lets fix a wrong publishing year for one specific book
with connection:
    cursor.execute("UPDATE books SET year = ? WHERE title = ?", (1950, "I, Robot"))
print("rows changed by the update:", cursor.rowcount)  # rowcount tells us how many rows were actually affected

# verify the change with a SELECT
cursor.execute("SELECT title, year FROM books WHERE title = ?", ("I, Robot",))
print("after update:", cursor.fetchone())

# DELETE - lets remove one book we no longer want in the table
with connection:
    cursor.execute("DELETE FROM books WHERE title = ?", ("Mort",))
print("rows deleted:", cursor.rowcount)

# COUNT(*) is a handy way to check how many rows are left afterwards
remaining = cursor.execute("SELECT COUNT(*) FROM books").fetchone()[0]
print("books remaining in the table:", remaining)

rows changed by the update: 1
after update: ('I, Robot', 1950)
rows deleted: 1
books remaining in the table: 5


## combining tables with JOIN
* this is where relational databases really shine: `authors` and `books` are two separate tables, connected only through `books.author_id` pointing at `authors.id`
* a `JOIN` combines matching rows from both tables into one result, so we can e.g. print a books title together with its authors name without ever pulling the two tables into python ourselves and matching them up by hand

In [7]:
# INNER JOIN books to authors, matching where books.author_id equals authors.id
cursor.execute('''
    SELECT books.title, books.year, authors.name
    FROM books
    JOIN authors ON books.author_id = authors.id   -- the actual "link" between the two tables
    ORDER BY authors.name, books.year
''')

for title, year, author_name in cursor:
    print(f"{title} ({year}) - by {author_name}")

# note: we could get this exact same information by writing three separate SELECTs and matching things up
# in python with loops/dicts ourselves - but the JOIN lets the database engine do that matching for us,
# which is both way less code and (for real, bigger datasets) much faster

I, Robot (1950) - by Isaac Asimov
Foundation (1951) - by Isaac Asimov
Guards! Guards! (1989) - by Terry Pratchett
The Left Hand of Darkness (1969) - by Ursula K. Le Guin
The Dispossessed (1974) - by Ursula K. Le Guin


## sqlite3.Row - accessing results by column name
* by default, every row we fetch is a plain tuple - we can only access columns by numeric position (`row[0]`, `row[1]`, ...)
* setting `connection.row_factory = sqlite3.Row` changes that: rows now behave like a dict-ish object, so we can access columns by **name** (`row["title"]`) as well as by index - much more readable, and it survives us reordering columns in the SELECT later

In [8]:
connection.row_factory = sqlite3.Row  # from now on, every query on this connection returns sqlite3.Row objects

# we need a fresh cursor to pick up the new row_factory (an already-open cursor keeps the old behaviour)
named_cursor = connection.cursor()
named_cursor.execute('''
    SELECT books.title, books.year, authors.name AS author_name
    FROM books
    JOIN authors ON books.author_id = authors.id
    ORDER BY books.year
''')

for row in named_cursor:
    # now we can access columns by name, no need to remember "title is index 0, year is index 1..."
    print(row["title"], "-", row["author_name"], f"({row['year']})")

# sqlite3.Row also still supports index access and even behaves a bit like a tuple
first_row = named_cursor.execute("SELECT title, year FROM books LIMIT 1").fetchone()
print("by name:", first_row["title"], "| by index:", first_row[0])
print("available column names:", first_row.keys())

I, Robot - Isaac Asimov (1950)
Foundation - Isaac Asimov (1951)
The Left Hand of Darkness - Ursula K. Le Guin (1969)
The Dispossessed - Ursula K. Le Guin (1974)
Guards! Guards! - Terry Pratchett (1989)
by name: The Left Hand of Darkness | by index: The Left Hand of Darkness
available column names: ['title', 'year']


## a quick note on ORMs
* writing raw SQL strings (like we did all lesson) works great and is worth understanding deeply - but in bigger real projects, people often reach for an **ORM** (Object-Relational Mapper) instead
* the most common python ORM is **SQLAlchemy** - the idea is that you define a normal python class (e.g. `class Book:`) and the ORM maps it directly to a database table, so most of the time you write python (`book.title`, `session.add(book)`) instead of raw SQL strings
* the tradeoff: its more convenient and often more readable/maintainable, but its also **another abstraction layer** you need to learn on top of SQL itself - and when something goes wrong or you need a very specific/optimized query, you often still need to understand the SQL underneath anyway
* we wont install or run SQLAlchemy in this lesson (remember: no pip installs, purely standard library!) - just keep it in mind as the natural "whats next" once raw `sqlite3` starts feeling repetitive

## quick comparison to other languages
* python's `sqlite3` follows the **DB-API 2.0** standard (`connect()`, `cursor()`, `execute()`, `fetchall()`) - third-party drivers for PostgreSQL (`psycopg2`) or MySQL (`mysql-connector-python`) use the exact same shape of interface, so switching database engines in python often just means swapping the driver
* this is conceptually very similar to Java's **JDBC** (`Connection`, `Statement`/`PreparedStatement`, `ResultSet`), C#'s **ADO.NET** (`SqlConnection`, `SqlCommand`, `SqlDataReader`), or the various database driver packages in Node.js - all of them center around the same connect -> prepare statement -> execute -> iterate results pattern
* parameterized queries exist in every mainstream language's database API specifically to prevent SQL injection: python's `?` placeholders, C#'s `@param` names in `SqlCommand`, Java's `?` placeholders in `PreparedStatement` - the concept (never glue user input directly into a SQL string) is universal, only the placeholder syntax differs slightly
* ORMs are extremely common outside python too: SQLAlchemy parallels Java's **Hibernate/JPA**, C#'s **Entity Framework**, JavaScript/TypeScript's **Sequelize**, **Prisma** or **TypeORM**, and Rust's **Diesel** - every ecosystem has its own "map a class to a table" answer

## Exercises
1. create a `tasks` table with columns `id` (primary key), `title` (text) and `done` (integer, 0 or 1), then insert 3 tasks of your choice (some done, some not)
2. write a query that selects only the tasks where `done` is `0` (not finished yet) and print them
3. update one task to `done = 1` by its `id`, then verify the change with a `SELECT`
4. delete one task by its `id`, then confirm the new row count afterwards with `SELECT COUNT(*)`

In [9]:
# TODO: solve exercise 1 here (create tasks table, insert 3 tasks)


# TODO: solve exercise 2 here (select tasks where done = 0)


# TODO: solve exercise 3 here (update a task to done = 1, then verify)


# TODO: solve exercise 4 here (delete a task, then confirm row count)

### sample solutions
* try the exercises above yourself first, only look here if you are stuck or want to compare your approach

In [10]:
# sample solution 1 - create the tasks table and insert 3 tasks
with connection:
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS tasks (
            id INTEGER PRIMARY KEY,   -- auto-incrementing unique id
            title TEXT NOT NULL,       -- what the task is about
            done INTEGER NOT NULL      -- 0 = not finished yet, 1 = finished
        )
    ''')
    cursor.executemany(
        "INSERT INTO tasks (title, done) VALUES (?, ?)",
        [
            ("write the sqlite lesson", 1),   # already done
            ("buy groceries", 0),               # not done yet
            ("water the plants", 0),            # not done yet
        ],
    )
print("tasks inserted")

# sample solution 2 - select only the tasks that are not done yet
cursor.execute("SELECT id, title FROM tasks WHERE done = 0")
open_tasks = cursor.fetchall()
print("open tasks:", open_tasks)

# sample solution 3 - mark one task as done, by its id, then verify
task_id_to_finish = open_tasks[0][0]  # lets just finish the first open task we found above
with connection:
    cursor.execute("UPDATE tasks SET done = 1 WHERE id = ?", (task_id_to_finish,))
cursor.execute("SELECT title, done FROM tasks WHERE id = ?", (task_id_to_finish,))
print("after update:", cursor.fetchone())  # done should now be 1

# sample solution 4 - delete a task by id, then confirm the row count
with connection:
    cursor.execute("DELETE FROM tasks WHERE id = ?", (task_id_to_finish,))
remaining_tasks = cursor.execute("SELECT COUNT(*) FROM tasks").fetchone()[0]
print("tasks remaining after delete:", remaining_tasks)

# we are done with the whole lesson now, so lets close the connection properly
# (its an in-memory database anyway, so it just vanishes completely - no file to clean up!)
connection.close()
print("connection closed")

tasks inserted
open tasks: [(2, 'buy groceries'), (3, 'water the plants')]
after update: ('buy groceries', 1)
tasks remaining after delete: 2
connection closed


* congratulation you finished this lesson

## what we learned
* what a relational database is (tables, rows, columns, primary keys) and why we'd reach for one over a plain csv/json file
* connecting with `sqlite3.connect()`, using a `cursor`, and the difference between manual `.commit()` and the automatic commit of `with connection:`
* creating tables with `CREATE TABLE IF NOT EXISTS` and typed columns
* inserting rows safely with parameterized `?` placeholders (and why f-strings + user input = SQL injection), plus bulk inserts with `executemany()`
* querying with `SELECT`/`WHERE`/`ORDER BY` and reading results with `.fetchone()`, `.fetchall()` or by iterating the cursor directly
* updating and deleting rows with `UPDATE`/`DELETE`, and checking affected rows with `.rowcount`
* combining related tables with `JOIN`
* `sqlite3.Row` for accessing query results by column name instead of numeric index
* that ORMs like SQLAlchemy exist as a higher-level alternative to writing raw SQL by hand